# [센서 실습 13] 토양 수분 센서 – 정전용량식 토양 수분 센서 v1.2

**정전용량식 토양 수분 센서**의 아날로그 출력을 ADC로 읽고, 건조·습윤 **2점 보정**으로 수분(%)을 구한 뒤, 릴레이로 펌프를 켜는 **스마트팜 자동 물주기**를 만듭니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
흙 속의 물은 전기를 저장하는 성질(**유전율**)이 공기보다 훨씬 큽니다 (공기 1, 흙 약 4, 물 약 80).
- 센서 기판의 전극이 콘덴서 역할을 하고, 주변 흙이 젖을수록 **정전용량이 커짐** → 출력 전압이 **낮아짐**
- 전극이 겉으로 드러나지 않아, 금속 전극이 녹스는 저항식 센서보다 **오래 사용**할 수 있음
- 흙의 종류마다 값이 다르므로 **보정**이 필요: 공기 중(완전 건조) 값과 물 속(완전 습윤) 값을 기록해 두고 그 사이를 0~100%로 나눔

$$\text{수분}(\%) = \frac{\text{건조값} - \text{측정값}}{\text{건조값} - \text{습윤값}} \times 100$$

### 부품 사양
| 항목 | 내용 |
|---|---|
| 출력 | 아날로그 약 1.2V(습윤) ~ 2.8V(건조) |
| 전원 | 3.3 ~ 5.5V |
| 방식 | 정전용량식 (부식에 강함) |
| 측정 깊이 | 센서 표시선까지 꽂음 |

## 2. 라즈베리파이 연결 방법
**준비물**: 정전용량식 토양 수분 센서 v1.2, MCP3008, 1채널 릴레이 모듈, 5V 소형 워터 펌프, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| 센서 VCC / GND | 3.3V / GND |  |
| 센서 AOUT | MCP3008 CH0 (1번) |  |
| 릴레이 VCC / GND | 5V / GND |  |
| 릴레이 IN | GPIO18 (12번 핀) | LOW에서 동작하는 모듈이 많음 |
| 펌프 | 릴레이 COM–NO 사이에 전원과 직렬 | 펌프 전원은 별도 공급 권장 |



**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
**2점 보정** 값으로 ADC 값을 수분(%)으로 바꿉니다. 보정값은 각자의 센서와 흙으로 측정해 바꾸세요.

In [ ]:
DRY_VALUE = 0.82     # 공기 중에서 읽은 ADC 값 (완전 건조, 보정값)
WET_VALUE = 0.38     # 물컵에 담갔을 때 읽은 ADC 값 (완전 습윤, 보정값)

def adc_to_moisture(adc_value):
    pct = (DRY_VALUE - adc_value) / (DRY_VALUE - WET_VALUE) * 100
    return round(float(np.clip(pct, 0, 100)), 1)

for a in (0.82, 0.70, 0.60, 0.38):
    print(f"ADC {a:.2f} → 수분 {adc_to_moisture(a)}%")

### 4-2. 실제 센서 클래스
센서 읽기와 함께 펌프를 켜는 `water()` 메서드를 가집니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealSoilSensor:
    def __init__(self, channel=0, pump_pin=18):
        from gpiozero import MCP3008, OutputDevice
        self.adc = MCP3008(channel=channel)
        self.pump = OutputDevice(pump_pin, active_high=False, initial_value=False)

    def read(self):
        v = self.adc.value
        return {"adc": round(v, 4), "moisture": adc_to_moisture(v), "pump_s": 0}

    def water(self, seconds):
        self.pump.on(); time.sleep(seconds); self.pump.off()

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 화분 하나, 14일 1시간 간격 | 베란다 화분 |
| 증발 | 낮(9~17시) 시간당 0.9%, 밤 0.2% | 햇빛·기온에 따라 마름 |
| 물주기 효과 | 펌프 1초당 수분 +6% (최대 85%) | 소형 펌프 유량 |
| 센서 특성 | 수분 0% ↔ ADC 0.82, 100% ↔ 0.38, 잡음 ±0.004 | 보정값과 같은 범위 |

가상 화분은 `water()`를 호출하면 **실제로 수분이 올라가므로**, 제어 로직이 제대로 동작하는지 확인할 수 있습니다.

In [ ]:
class VirtualSoilSensor:
    def __init__(self, dt=3600, seed=13):
        self.dt, self.t, self.m = dt, 0.0, 60.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        hour = (self.t / 3600) % 24
        self.m -= (0.9 if 9 <= hour < 17 else 0.2) * self.dt / 3600     # 증발
        self.m = max(self.m, 0.0)
        adc = DRY_VALUE - self.m / 100 * (DRY_VALUE - WET_VALUE) + self.rng.normal(0, 0.004)
        self.t += self.dt
        return {"adc": round(adc, 4), "moisture": adc_to_moisture(adc), "pump_s": 0}

    def water(self, seconds):
        self.m = min(85.0, self.m + 6 * seconds)                         # 물 주기

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualSoilSensor(dt=3600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1시간 간격으로 14일 동안 측정하며, **수분이 30% 미만이면 펌프를 5초 동작**시킵니다. (수집 반복문 안에 제어 코드가 들어 있습니다.)

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 336               # 측정 횟수
INTERVAL = 3600               # 측정 간격(초)

sensor = RealSoilSensor() if USE_REAL_SENSOR else VirtualSoilSensor(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if value["moisture"] < 30:              # 30% 미만이면 자동 물주기
        sensor.water(5)
        records[-1]["pump_s"] = 5
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
물을 준 횟수와 시각, 하루 평균 수분을 확인합니다.

In [ ]:
df["day"] = df["time_s"] / 86400
w = df[df["pump_s"] > 0]
print(f"자동 물주기: 총 {len(w)}회, 펌프 동작 {int(w['pump_s'].sum())}초")
for _, r in w.iterrows():
    d = r["time_s"] / 3600
    print(f"  {int(d // 24) + 1}일차 {int(d % 24):02d}시  수분 {r['moisture']:.1f}% → 펌프 {int(r['pump_s'])}초")
print(f"평균 수분: {df['moisture'].mean():.1f}%, 최저 {df['moisture'].min():.1f}%")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(df["day"], df["moisture"], "k-", lw=1.2, label="토양 수분")
plt.axhline(30, color="gray", ls="--", label="물주기 기준 30%")
plt.plot(w["time_s"] / 86400, w["moisture"], "kv", ms=8, label="펌프 동작")
plt.title("토양 수분과 자동 물주기 (14일)"); plt.xlabel("경과 일수"); plt.ylabel("수분 (%)")
plt.ylim(0, 100); plt.grid(alpha=0.3); plt.legend(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "토양수분센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "정전용량식 토양 수분 센서 v1.2",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/soil", payload)
# client.disconnect()

## 직접 해 보기
- 물주기 기준을 40%로, 펌프 시간을 3초로 바꾸면 물주기 횟수가 어떻게 바뀌나요?
- 밤(20시~6시)에는 물을 주지 않도록 조건을 추가해 보세요.
- 자신의 센서로 공기 중·물속 값을 측정해 `DRY_VALUE`, `WET_VALUE`를 보정하는 방법을 정리해 보세요.